In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from nltk.corpus import stopwords
from nltk import *
from sklearn import *
import string
import json
import glob
import re
import csv
import bs4
import contractions
import os
from docx import Document
from pptx import Presentation
import requests
import PyPDF2
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
from typing import List
import nltk
import matplotlib.pyplot as plt

In [2]:
def CheckFileType(files_path: List[str]) -> List[str]: 
    ''' 
    Takes a list of files paths in a list form and returns a list of Extensions of Files
    '''
    assert isinstance(files_path, list), "files_path must be a list."
    assert all(isinstance(path, str) for path in files_path), "Each element in files_path must be a string."
    file_path_extension = []
    for path in files_path:
        file_name,file_extension = os.path.splitext(path)
        file_path_extension.append(file_extension)
    return file_path_extension

In [3]:
def LoadTextDataToTextFile(files_path,output_file,text_column=None, url=None):
    
    assert isinstance(files_path, list), "files_path must be a List."
    assert isinstance(output_file, str), "output_file must be a String."
    
    
    for extension,file_path in zip(CheckFileType(files_path),files_path):
        
        '''Here we are using CheckFileType function to return extensions and 
           then iterating through both list using zip functions.
           We load all text from any of the documents to a single (.txt) file
           
           Parameters:
                      files_path: List of all the document files' path -> List(str)
                      output_file: output file path -> str
                      text_coulmn: csv file, must be included -> int
                      url: must be used only for url -> str
           '''
        
        file_name, file_extension = os.path.splitext(file_path) # As CheckFileType wasn't been accessible, using this to seperate file_name, file_extension 
        
        # load txt file
        if extension== '.txt':
            try:
                file_name,file_extension = os.path.splitext(file_path)
                with open(file_path, 'r', encoding='utf-8') as infile, open(output_file,'a', encoding='utf-8') as outfile: # "append" to output_file 
                    data = infile.readline()
                    text_split = data.split('.')
                    outfile.write(f"**_{file_name}.{extension}Text is been Entered:_**\n" )
                    for data_new in text_split:
                        outfile.write(data_new)
                print(f"Contents from {file_path} have been copied to {output_file}.")
            except FileNotFoundError:
                print(f"Error: The file {file_path} does not exist.")
            except Exception as e:
                print(f"An error occurred: {e}")
                

        # Load csv file
        elif extension == '.csv':
            try:
                if text_column is None:
                    raise ValueError("For CSV files, 'text_column' must be specified.")
                with open(file_path,'r',encoding='utf-8') as infile, open(output_file,'a', encoding='utf-8') as outfile: 
                    data = pd.read_csv(infile)
                    outfile.write(f"**_{file_name}.{extension}Text is been Entered:_**\n" )
                    for row in data:
                        outfile.write(row)
                print(f"Contents from {file_path} have been copied to {output_file}.")
            except FileNotFoundError:
                print(f"Error: The file {file_path} does not exist.")
            except Exception as e:
                print(f"An error occurred: {e}")

        # load json file
        elif extension == '.json':
            try:
                with open(file_path, 'r', encoding='utf-8') as infile, open(output_file,'a',encoding='utf-8') as outfile:
                    data = json.load(infile)
                    outfile.write(f"**_{file_name}.{extension}Text is been Entered:_**\n" )
                    if isinstance(data, list):
                        for item in data:
                            if 'text' in item:
                                outfile.write(item['text'] + '\n')
                    elif isinstance(data, dict):
                        json.dump(data, outfile, indent=4, ensure_ascii=False)
                        outfile.write('\n')
                    else:
                        raise ValueError("JSON file structure not supported. It should be a list of objects with a 'text' key.")
                print(f"Contents from {file_path} have been copied to {output_file}.")
            except FileNotFoundError:
                print(f"Error: The file {file_path} does not exist.")
            except json.JSONDecodeError:
                print(f"Error: The file {file_path} is not a valid JSON file.")
            except Exception as e:
                print(f"An error occurred: {e}")

        # Load word(.docx) file
        elif extension == '.docx':
            file_name, file_extension = os.path.splitext(file_path)
            doc = Document(file_path)
            txt_filename_docs = f"{output_file}"
            with open(txt_filename_docs, 'a', encoding='utf-8') as outfile:
                outfile.write(f"**_{file_name}.{extension}Text is been Entered:_**\n" )
                for para in doc.paragraphs:
                    outfile.write(para.text + '\n')
            

        # Load powerpoint(.pptx) file
        elif extension == '.pptx':
            file_name, file_extension = os.path.splitext(file_path)
            prs = Presentation(file_path)
            txt_filename_ppt = f'{output_file}'
            with open(txt_filename_ppt, 'a', encoding='utf-8') as outfile:
                for slide in prs.slides:
                    for shape in slide.shapes:
                        if hasattr(shape, "text"):
                            outfile.write(shape.text + '\n')

        elif extension == '.pdf':
            with open(file_path, 'rb') as file:
                reader = PyPDF2.PdfReader(file)
                text_data_pdf = []
                with open(output_file, 'a', encoding='utf-8') as outfile:
                    outfile.write(f"**_{file_name}.{extension} Text has been Entered:_**\n")
                    for page in reader.pages:
                        text = page.extract_text()
                        if text:
                            outfile.write(text + '\n')

        # Load text data from a website
        elif url:
            response = requests.get(url)
            if response.status_code == 200:
                soup = BeautifulSoup(response.content, 'html.parser')
                paragraphs = soup.find_all('p')
                text_data_url = [p.get_text() for p in paragraphs if p.get_text().strip()]
                
                with open(output_file, 'a', encoding='utf-8') as outfile:
                    outfile.write(f"**_Content from URL {url} has been Entered:_**\n")
                    for text in text_data_url:
                        outfile.write(text + '\n')
            else:
                raise ValueError("Failed to retrieve the website content.")
        else:
            raise ValueError("Unsupported file type. Supported types are 'txt', 'csv', and 'json'.")

        return output_file

In [4]:
def WriteTextData(data, file_path): 
    ''' Used for the last part of execution of Expressing the data 
        can be to a ppt or a word or any other document , 
        this needs more change on how to arrange the data in that file 
                                                                     '''
    file_name, file_extension = os.path.splitext(file_path)
    
    if file_extension == '.txt':
        with open(file_path, 'w', encoding='utf-8') as file:
            file.write('\n'.join(data))
    
    elif file_extension == '.csv':
        if not isinstance(data, list):
            raise ValueError("Data must be a list to write to a CSV file.")
        df = pd.DataFrame(data, columns=['text'])
        df.to_csv(file_path, index=False, encoding='utf-8')
    
    elif file_extension == '.json':
        if not isinstance(data, list):
            raise ValueError("Data must be a list to write to a JSON file.")
        json_data = [{'text': text} for text in data]
        with open(file_path, 'w', encoding='utf-8') as file:
            json.dump(json_data, file, ensure_ascii=False, indent=4)
    
    elif file_extension == '.docx':
        doc = Document()
        for text in data:
            doc.add_paragraph(text)
        doc.save(file_path)
    
    elif file_extension == '.pptx':
        prs = Presentation()
        slide_layout = prs.slide_layouts[5]
        for text in data:
            slide = prs.slides.add_slide(slide_layout)
            textbox = slide.shapes.add_textbox(left=0, top=0, width=prs.slide_width, height=prs.slide_height)
            text_frame = textbox.text_frame
            text_frame.text = text
        prs.save(file_path)
    
    elif file_extension == '.pdf':
        c = canvas.Canvas(file_path, pagesize=letter)
        width, height = letter
        y = height - 40
        for text in data:
            if y < 40:
                c.showPage()
                y = height - 40
            c.drawString(40, y, text)
            y -= 20
        c.save()

    else:
        raise ValueError(f"File extension '{file_extension}' is not supported.")


In [5]:
def DownloadTextFromURL(url):
    response = requests.get(url)
    if response.status_code == 200:
        return response.text.splitlines()
    else:
        raise ValueError("Failed to retrieve the URL content.")

In [6]:
def CleanText(files_path, output_file, cleaned_output_file):
    """This Function is part of text preprocessing and cleaning
        includes removing certain regex patterns, dates, days, months, stopwords,stemming, lemminaization, punctuation,removing numbers 
        """
    text = ""
    with open(output_file, 'r', encoding='utf-8') as file:
        text = file.read()
    patterns = [
    r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}',  # Email
    r'(\+?\d{1,4}[\s-])?(?:\(\d{1,3}\)[\s-]?)?\d{1,4}[\s-]?\d{1,4}[\s-]?\d{1,9}',  # Phone numbers
    r'\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b',  # Dates
    r'https?://[^\s/$.?#].[^\s]*',  # URLs
    r'\[(\d+)\]',  # Bracketed numbers
    r'10.\d{4,9}/[-._;()/:A-Z0-9]+',  # DOIs
    r'(?:ISBN(?:-13)?:?\s*)?(?=[-0-9]{13}$|(?=(?:[-0-9]{17}$)|(?:[-0-9X]{10}$))(?:97[89][-0-9]{10}$))\d{1,5}[-\s]?\d{1,7}[-\s]?\d{1,7}[-\s]?\d{1,7}[-\s]?(?:\d|X)',  # ISBNs
    r'[-+]?\d*\.?\d+([eE][-+]?\d+)?',  # Numbers and scientific notation
    r'>.*\n([A-Z\n]+)',  # Block caps
    r'\$\$.*?\$\$',  # LaTeX equations
    r'^([a-zA-Z0-9_\-]+)\.([a-zA-Z0-9]+)Text\sis\sbeen\sEntered:\s:\s$',  # Custom pattern
    r'#\w+',  # Hashtags
    r'@\w+',  # Mentions
    r'\b(?:\d{1,2} [A-Za-z]{3,9} \d{4}|\d{4}/\d{2}/\d{2})\b',  # Dates (expanded formats)
    r'\b(?:\d{1,3}\.){3}\d{1,3}\b',  # IP addresses
    r'<.*?>',  # HTML tags
    r'\((.*?)\)',  # Bracketed content
    r'\b(?:\$|€|₹|£)\d+(?:\.\d{1,2})?\b',  # Currency values
    r'[^\x00-\x7F]+',  # Non-ASCII characters
    r'\b(?:\d{4}[- ]?){3}\d{4}\b',  # Credit card numbers
    r'\b[0-9A-Fa-f]{8}-[0-9A-Fa-f]{4}-[0-9A-Fa-f]{4}-[0-9A-Fa-f]{4}-[0-9A-Fa-f]{12}\b',  # UUIDs
    r'(.)\1{2,}',  # Repeated characters
    r'(?:[A-Za-z]:)?(?:\\[A-Za-z0-9_.-]+)+\\?',  # File paths
    r'\b0[xX][0-9a-fA-F]+\b',  # Hexadecimal numbers
    r'\b[A-Z]+\b',  # All caps
    r'["\'](.*?)["\']',  # Quoted strings
    r'\b[A-Z][a-z]*\b'  # Extract capitalized words
]

    for pattern in patterns:
        text = re.sub(pattern, '', text) # remove regex patterns

    words = text.split()
    months = ['january', 'february', 'march', 'april', 'may', 'june', 'july', 'august', 'september', 'october', 'november', 'december'] 
    stops = set(stopwords.words('english'))
    stops.update(months) # combing stopwords and months from list to remove them collectively

    # Initialize the stemmer and lemmatizer
    stemmer = PorterStemmer()                          # converts a word to its stem form like running to run 
    lemmatizer = WordNetLemmatizer()                   # converts a word to its dictionary form for no duplicates 

    final = [lemmatizer.lemmatize(stemmer.stem(word)) for word in words if word.lower() not in stops]

    final_text = " ".join(final)
    final_text = final_text.translate(str.maketrans("", "", string.punctuation))
    final_text = "".join([i for i in final_text if not i.isdigit()])
    while "  " in final_text:
        final_text = final_text.replace("  ", " ")

    # Write to a  cleaned text to the new output file
    with open(cleaned_output_file, 'w', encoding='utf-8') as file:
        file.write(final_text)

    return final_text


In [7]:
def LoadTextData(file): # Just in case of error fuction , need no change or modification , just to scout
    with open(file, "r", encoding="utf-8") as f:
        data = f.readlines()
    return data

In [8]:
files_path = [
    "C:\\Projects\\Project SB\\TextModeling\\Sources\\AIML Panthers_2024_Group 85.pdf",
    "C:\\Projects\\Project SB\\TextModeling\\Sources\\hellowether1.csv",
    "C:\\Projects\\Project SB\\TextModeling\\Sources\\Ideation.docx"
]

output_file = r"C:\\Projects\\Project SB\\TextModeling\\output\\output.txt"
cleaned_output_file = r"C:\\Projects\\Project SB\\TextModeling\\output\\CleanedText1.txt"

# Generate the output.txt file
LoadTextDataToTextFile(files_path, output_file)

# Clean the text and generate the cleaned_output_file
cleaned_txtfile = CleanText(files_path, output_file, cleaned_output_file)
